In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import classification_report, roc_auc_score, roc_curve, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 6)
sns.set_style('whitegrid')

In [ ]:
df = pd.read_csv('data:/data.csv')
print(df.shape)
df.head(10)

**[Buraya kendi yorumunu ekle — veriyi ilk gördüğünde ne fark ettin?]**

In [ ]:
df.info()
df.isnull().sum()

In [ ]:
df.describe()

**[Buraya kendi yorumunu ekle — temel istatistiklerden ne öğrendin?]**

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

axes[0, 0].hist(df['age'], bins=30, color='steelblue', edgecolor='white')
axes[0, 0].set_title('Age')

axes[0, 1].hist(df['balance'], bins=30, color='seagreen', edgecolor='white')
axes[0, 1].set_title('Balance')

axes[0, 2].hist(df['estimated_salary'], bins=30, color='darkorange', edgecolor='white')
axes[0, 2].set_title('Estimated Salary')

axes[1, 0].hist(df['credit_score'], bins=30, color='mediumpurple', edgecolor='white')
axes[1, 0].set_title('Credit Score')

axes[1, 1].bar(df['tenure'].value_counts().index, df['tenure'].value_counts().values, color='tomato')
axes[1, 1].set_title('Tenure')

axes[1, 2].pie(df['churn'].value_counts(), labels=['Retained', 'Churned'],
               autopct='%1.1f%%', colors=['steelblue', 'tomato'])
axes[1, 2].set_title('Churn')

plt.tight_layout()
plt.savefig('figures/distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
numeric_cols = ['credit_score', 'age', 'tenure', 'balance', 'products_number', 'estimated_salary', 'churn']
plt.figure(figsize=(10, 8))
sns.heatmap(df[numeric_cols].corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0, linewidths=0.5)
plt.title('Korelasyon Matrisi')
plt.tight_layout()
plt.savefig('figures/correlation.png', dpi=150, bbox_inches='tight')
plt.show()

**[Buraya kendi yorumunu ekle — hangi değişkenler churn ile en çok ilişkili?]**

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

df.groupby('country')['churn'].mean().plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Churn Rate by Country')
axes[0].set_ylabel('Churn Rate')
axes[0].tick_params(axis='x', rotation=0)

df.groupby('gender')['churn'].mean().plot(kind='bar', ax=axes[1], color='seagreen')
axes[1].set_title('Churn Rate by Gender')
axes[1].tick_params(axis='x', rotation=0)

df.groupby('products_number')['churn'].mean().plot(kind='bar', ax=axes[2], color='tomato')
axes[2].set_title('Churn Rate by Products')
axes[2].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig('figures/churn_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
data = df.drop('customer_id', axis=1).copy()

le = LabelEncoder()
data['country'] = le.fit_transform(data['country'])
data['gender'] = le.fit_transform(data['gender'])

X = data.drop('churn', axis=1)
y = data['churn']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

print(f'Train: {X_train.shape[0]} | Test: {X_test.shape[0]}')

In [ ]:
rf = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

gb = GradientBoostingClassifier(n_estimators=200, learning_rate=0.05, max_depth=5, random_state=42)
gb.fit(X_train, y_train)

for name, model, X_eval in [('Random Forest', rf, X_test), ('Gradient Boosting', gb, X_test)]:
    auc = roc_auc_score(y_test, model.predict_proba(X_eval)[:, 1])
    print(f'\n=== {name} | AUC: {auc:.4f} ===')
    print(classification_report(y_test, model.predict(X_eval)))

**[Buraya kendi yorumunu ekle — hangi model daha iyi performans gösterdi ve neden?]**

In [ ]:
plt.figure(figsize=(10, 7))
for name, model, X_eval in [('Random Forest', rf, X_test), ('Gradient Boosting', gb, X_test)]:
    fpr, tpr, _ = roc_curve(y_test, model.predict_proba(X_eval)[:, 1])
    auc = roc_auc_score(y_test, model.predict_proba(X_eval)[:, 1])
    plt.plot(fpr, tpr, lw=2, label=f'{name} (AUC={auc:.3f})')
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('FPR')
plt.ylabel('TPR')
plt.title('ROC Curves')
plt.legend()
plt.tight_layout()
plt.savefig('figures/roc.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
feat_imp = pd.DataFrame({'feature': X_train.columns, 'importance': rf.feature_importances_})
feat_imp = feat_imp.sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(data=feat_imp, x='importance', y='feature', palette='Blues_d')
plt.title('Feature Importances')
plt.tight_layout()
plt.savefig('figures/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
df_ltv = df.copy()
full_X = data.drop('churn', axis=1)

df_ltv['churn_prob'] = rf.predict_proba(full_X)[:, 1]
df_ltv['retention_prob'] = 1 - df_ltv['churn_prob']

DISCOUNT_RATE = 0.10
YEARS = 5

df_ltv['annual_revenue'] = (
    df_ltv['balance'] * 0.02 +
    df_ltv['products_number'] * 500 +
    df_ltv['estimated_salary'] * 0.005
)

def calculate_ltv(row):
    ltv = 0
    survival = 1.0
    for t in range(1, YEARS + 1):
        survival *= row['retention_prob']
        ltv += (row['annual_revenue'] * survival) / ((1 + DISCOUNT_RATE) ** t)
    return ltv

df_ltv['ltv'] = df_ltv.apply(calculate_ltv, axis=1)

print('LTV Özeti:')
print(df_ltv['ltv'].describe())

**[Buraya kendi yorumunu ekle — LTV dağılımı hakkında ne düşünüyorsun?]**

In [ ]:
df_ltv['ltv_segment'] = pd.qcut(df_ltv['ltv'], q=4, labels=['Bronze', 'Silver', 'Gold', 'Platinum'])

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].hist(df_ltv['ltv'], bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('LTV Dağılımı')
axes[0].set_xlabel('LTV ($)')

counts = df_ltv['ltv_segment'].value_counts()
axes[1].pie(counts, labels=counts.index, autopct='%1.1f%%',
            colors=['#CD7F32', '#C0C0C0', '#FFD700', '#E5E4E2'])
axes[1].set_title('LTV Segment Dağılımı')

plt.tight_layout()
plt.savefig('figures/ltv_dist.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
segment_order = ['Bronze', 'Silver', 'Gold', 'Platinum']
colors = ['#CD7F32', '#C0C0C0', '#FFD700', '#E5E4E2']

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

sns.boxplot(data=df_ltv, x='ltv_segment', y='ltv', order=segment_order, palette=colors, ax=axes[0, 0])
axes[0, 0].set_title('LTV by Segment')

sns.boxplot(data=df_ltv, x='ltv_segment', y='balance', order=segment_order, palette=colors, ax=axes[0, 1])
axes[0, 1].set_title('Balance by Segment')

churn_by_seg = df_ltv.groupby('ltv_segment')['churn_prob'].mean().reindex(segment_order)
axes[1, 0].bar(segment_order, churn_by_seg.values, color=colors, edgecolor='gray')
axes[1, 0].set_title('Avg Churn Probability by Segment')

country_ltv = df_ltv.groupby('country')['ltv'].mean().sort_values(ascending=False)
axes[1, 1].bar(country_ltv.index, country_ltv.values, color=['steelblue', 'seagreen', 'tomato'])
axes[1, 1].set_title('Average LTV by Country')

plt.tight_layout()
plt.savefig('figures/ltv_segments.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
segment_summary = df_ltv.groupby('ltv_segment').agg(
    musteri_sayisi=('customer_id', 'count'),
    ort_ltv=('ltv', 'mean'),
    median_ltv=('ltv', 'median'),
    ort_balance=('balance', 'mean'),
    ort_yas=('age', 'mean'),
    ort_urun=('products_number', 'mean'),
    gercek_churn=('churn', 'mean'),
    tahmin_churn=('churn_prob', 'mean')
).round(2)

segment_summary

**[Buraya kendi yorumunu ekle — segment tablosundan ne çıkarıyorsun? Hangi segment iş açısından en kritik?]**

In [ ]:
plt.figure(figsize=(12, 7))
sc = plt.scatter(df_ltv['churn_prob'], df_ltv['ltv'],
                 c=df_ltv['balance'], cmap='viridis', alpha=0.4, s=15)
plt.colorbar(sc, label='Balance ($)')
plt.xlabel('Churn Probability')
plt.ylabel('LTV ($)')
plt.title('LTV vs Churn Probability')
plt.tight_layout()
plt.savefig('figures/ltv_vs_churn.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
at_risk = df_ltv[
    (df_ltv['ltv_segment'] == 'Platinum') & (df_ltv['churn_prob'] > 0.5)
].sort_values('ltv', ascending=False)

print(f'Risk Altındaki Platinum Müşteri: {len(at_risk)}')
print(f'Risk Altındaki Toplam LTV: ${at_risk["ltv"].sum():,.0f}')

at_risk[['customer_id', 'country', 'age', 'balance', 'products_number',
          'churn_prob', 'ltv']].head(20)

In [ ]:
total = df_ltv['ltv'].sum()
plat = df_ltv[df_ltv['ltv_segment'] == 'Platinum']['ltv'].sum()
risk = at_risk['ltv'].sum()

print(f'Toplam Portföy LTV  : ${total:,.0f}')
print(f'Platinum LTV        : ${plat:,.0f} ({plat/total*100:.1f}%)')
print(f'Risk Altındaki LTV  : ${risk:,.0f} ({risk/total*100:.1f}%)')

print('\nÜlke Bazında LTV:')
for country, ltv in df_ltv.groupby('country')['ltv'].sum().sort_values(ascending=False).items():
    print(f'  {country}: ${ltv:,.0f} ({ltv/total*100:.1f}%)')

**[Buraya kendi yorumunu ekle — genel sonuçlar ve önerilen aksiyon planı nedir?]**

In [ ]:
output_cols = ['customer_id', 'country', 'gender', 'age', 'balance', 'products_number',
               'estimated_salary', 'churn', 'churn_prob', 'retention_prob',
               'annual_revenue', 'ltv', 'ltv_segment']

df_ltv[output_cols].to_csv('data:/ltv_results.csv', index=False)
print('Kaydedildi: data:/ltv_results.csv')
df_ltv[output_cols].head()